### This notebook read data from the bronze delta table and write it to gold schema as per the STAR schema

In [0]:
from pyspark.sql.functions import (col, concat, lit, current_date, year, months_between, 
                                   floor, min,max, to_date, date_add, lit, col,date_format,
                                   dayofmonth,date_format,dayofweek,weekofyear,month,quarter, length,unix_timestamp)

from datetime import timedelta

In [0]:
%sql
---Create gold schema 
CREATE SCHEMA IF NOT EXISTS gold

In [0]:
###Create dim rider
##1) Read the table
stg_riders_df = spark.read.table('bronze.stg_riders')

##2) Create dataframe
df = stg_riders_df.withColumn('name', concat(col('first'), lit(' '), col('last')))\
    .withColumn('age', floor(months_between(current_date(), col('birthday'))/12))\
    .withColumn('age_when_joined', floor(months_between(col('account_start_date'), col('birthday'))/12))\
    .drop('first','last','account_start_date','account_end_date')

## 3) WWrite to dimensional table rider
df.write.mode("overwrite").format('delta').saveAsTable('gold.dim_rider')
display(spark.read.table('gold.dim_rider'))


rider_id,address,birthday,is_member,name,age,age_when_joined
23344,0309 Cassie Fall Suite 266,1979-12-13,True,Curtis Harris,46,39
23345,55226 Brandon Forge Apt. 718,1990-07-25,True,Joshua Farmer,35,30
23346,616 Maurice Forges Suite 888,1988-03-26,True,Jill Rangel,38,32
23347,33245 Shane Walks,1973-10-10,True,Jamie Evans,52,45
23348,57751 English Lake,2004-01-22,True,Shirley Gomez,22,16
23349,976 Tracy Rest,1986-11-05,True,Michael Schultz,39,29
23350,554 Audrey Glen Apt. 976,1984-10-12,True,Nathan Baker,41,28
23351,886 Joanna Unions,1990-11-07,False,Shaun Casey,35,23
23352,3731 Hayes Rest Apt. 371,1994-08-18,True,Jennifer Day,31,24
23353,93349 Joshua Bridge,1962-12-08,True,Troy Gross,63,55


In [0]:
###Create dim station
##1) Read the table
stg_stations_df = spark.read.table('bronze.stg_stations')

##2) Create dataframe
df = stg_stations_df

## 3) Write to dimensional table rider
df.write.mode("overwrite").format('delta').saveAsTable('gold.dim_stations')
display(spark.read.table('gold.dim_stations'))


station_id,name,lattitude,longitude
525,Glenwood Ave & Touhy Ave,42.012701,-87.66605799999999
KA1503000012,Clark St & Lake St,41.88579466666667,-87.63110066666668
637,Wood St & Chicago Ave,41.895634,-87.672069
13216,State St & 33rd St,41.8347335,-87.6258275
18003,Fairbanks St & Superior St,41.89580766666667,-87.62025316666669
KP1705001026,LaSalle Dr & Huron St,41.894877,-87.632326
13253,Lincoln Ave & Waveland Ave,41.948797,-87.675278
KA1503000044,Rush St & Hubbard St,41.890173,-87.62618499999999
KA1504000140,Winchester Ave & Elston Ave,41.92403733333333,-87.67641483333334
TA1305000032,Clinton St & Madison St,41.882242,-87.64106600000001


In [0]:
###Create dim date table
#1) Compute min/max date from the trips table
row = (spark.read.table("bronze.stg_trips")
       .select(to_date("started_at").alias("started_date"))
       .agg(min("started_date").alias("min_date"),
            max("started_date").alias("max_date"))
       .first())

trips_start_date = row["min_date"]
trips_end_date   = row["max_date"]

# 2) Add one year to end date
end_plus_1yr = trips_end_date + timedelta(days=365)

# 3) Compute number of days
num_days = (end_plus_1yr - trips_start_date).days + 1


# 4) Build date dimension
df = (spark.range(0, num_days)
      .withColumnRenamed("id", "day_offset")
    .withColumn("date", date_add(lit(trips_start_date), col("day_offset").cast("int")))
    .withColumn("date_id", date_format("date", "yyyyMMdd").cast("int"))
          )

##Add other attributes
df = (df
      .withColumn("day", dayofmonth("date"))
      .withColumn("day_name", date_format("date", "EEEE"))
      .withColumn("day_of_week", dayofweek("date"))
      .withColumn("day_of_month", dayofmonth("date"))
      .withColumn("day_of_quarter", floor(dayofmonth("date")/7)+1)
      .withColumn("week", floor(dayofmonth("date")/4))
      .withColumn("month", month("date"))
      .withColumn("year", year("date")))

###Write table to gold schema
df.write.mode('overwrite').format('delta').SaveAsTable('gold.dim_date')

#display(spark.read.table('gold.dim_date'))

In [0]:
###Create fact trip

## 1) Read source + dimensions
st = spark.read.table("bronze.stg_trips").alias("st")
dd = spark.read.table("dim_datetime").alias("dd")
r  = spark.read.table("dim_rider").alias("r")
ds_start = spark.read.table("dim_station").alias("ds_start")
ds_end   = spark.read.table("dim_station").alias("ds_end")

## 2) Join dimension tables
joined_df = (st.join(dd, to_date(col("st.started_at")) == col("dd.date"), "inner")
    .join(r,  col("st.rider_id") == col("r.rider_id"), "inner")
    .join(ds_start, col("st.start_station_id") == col("ds_start.station_id"), "inner")
    .join(ds_end,   col("st.end_station_id") == col("ds_end.station_id"), "inner"))

# 3) Select + derive fact fields
fact_trips_df = (joined_df
    .select(
        st.trip_id.alias("trip_id"),
        col("dd.date_id").alias("date_id"),
        col("r.rider_id").alias("rider_id"),
        col("st.start_station_id").alias("start_station_id"),
        col("st.end_station_id").alias("end_station_id"),
        col("st.rideable_type").alias("rideable_type"),
        floor(months_between(col("st.started_at"), col("r.birthday")) / lit(12)).cast("int").alias("rider_age"),
        col("st.started_at").alias("started_at"),
        col("st.ended_at").alias("ended_at"),
        ((unix_timestamp(col("st.ended_at")) - unix_timestamp(col("st.started_at"))) / lit(60)).cast("int").alias("trip_duration"))
)


###Write to the gold schema
fact_trips_df.write.mode('overwrite').format('delta').SaveAsTable('gold.fact_trip')

# display(spark.read.table('gold.fact_trip'))

min(started_at)
2021-02-01


In [0]:
###Create Fact payment
sp = spark.read.table("bronze.stg_payments").alias("sp")  
fr = spark.read.table("dim_rider").alias("fr")
dd = spark.read.table("dim_date").alias("dd")         

# Join + select (fact_payment)
fact_payment_df = (
    sp
    .join(fr, col("sp.rider_id") == col("fr.rider_id"), "inner")
    .join(dd, to_date(col("sp.date")) == col("dd.date"), "inner")
    .select(
        col("sp.payment_id").alias("payment_id"),
        col("fr.rider_id").alias("rider_id"),
        col("dd.date_id").alias("date_id"),
        col("sp.amount").alias("amount")))


###Write to the gold schema
fact_payment_df.write.mode('overwrite').format('delta').SaveAsTable('gold.fact_payment')

# display(spark.read.table('gold.fact_payment'))